In [60]:
import pandas as pd
import numpy as np
import joblib
#
df = pd.read_csv('goalscorers.csv')
df =df.iloc[47601:]
df.drop(columns=["date"],inplace=True)
df.reset_index(drop=True , inplace=True)
df.shape
df.head(2)



,home_team,away_team,team,scorer,minute,own_goal,penalty
0,Mexico,South Africa,Mexico,Julián Quiñones,9.0,False,False
1,Mexico,South Africa,Mexico,Raúl Jiménez,67.0,False,False


In [45]:
df_2 = pd.read_csv('results.csv')
df_2 = df_2.iloc[49405:]
df_2.dropna(inplace=True)
df_2.drop(columns=['tournament','city','country','date'] , inplace=True)
df_2.reset_index(drop=True,inplace=True)

df_2.head(2)


,home_team,away_team,home_score,away_score,neutral
0,Mexico,South Africa,2.0,0.0,False
1,South Korea,Czech Republic,2.0,1.0,True


In [46]:
final_df = pd.merge(df,df_2 , on=["home_team","away_team"], how='inner')

In [47]:
final_df

,home_team,away_team,team,scorer,minute,own_goal,penalty,home_score,away_score,neutral
0,Mexico,South Africa,Mexico,Julián Quiñones,9.0,False,False,2.0,0.0,False
1,Mexico,South Africa,Mexico,Raúl Jiménez,67.0,False,False,2.0,0.0,False
2,South Korea,Czech Republic,Czech Republic,Ladislav Krejčí,59.0,False,False,2.0,1.0,True
3,South Korea,Czech Republic,South Korea,Hwang In-beom,67.0,False,False,2.0,1.0,True
4,South Korea,Czech Republic,South Korea,Oh Hyeon-gyu,80.0,False,False,2.0,1.0,True
...,...,...,...,...,...,...,...,...,...,...
84,Switzerland,Bosnia and Herzegovina,Switzerland,Johan Manzambi,74.0,False,False,4.0,1.0,True
85,Switzerland,Bosnia and Herzegovina,Switzerland,Rubén Vargas,84.0,False,False,4.0,1.0,True
86,Switzerland,Bosnia and Herzegovina,Switzerland,Johan Manzambi,90.0,False,False,4.0,1.0,True
87,Switzerland,Bosnia and Herzegovina,Switzerland,Granit Xhaka,90.0,False,True,4.0,1.0,True


In [48]:
final_df['home_scored'] = (final_df['team']==final_df['home_team']).astype(int)
final_df['away_scored'] = (final_df['team']==final_df['away_team']).astype(int)


In [49]:
final_df['home_scored'] = np.where(final_df['own_goal'] == True, 1 - final_df['home_scored'], final_df['home_scored'])
final_df['away_scored'] = np.where(final_df['own_goal'] == True, 1 - final_df['away_scored'], final_df['away_scored'])
final_df.drop(columns=['home_score', 'away_score'], inplace=True, errors='ignore')
final_df.head(80)

,home_team,away_team,team,scorer,minute,own_goal,penalty,neutral,home_scored,away_scored
0,Mexico,South Africa,Mexico,Julián Quiñones,9.0,False,False,False,1,0
1,Mexico,South Africa,Mexico,Raúl Jiménez,67.0,False,False,False,1,0
2,South Korea,Czech Republic,Czech Republic,Ladislav Krejčí,59.0,False,False,True,0,1
3,South Korea,Czech Republic,South Korea,Hwang In-beom,67.0,False,False,True,1,0
4,South Korea,Czech Republic,South Korea,Oh Hyeon-gyu,80.0,False,False,True,1,0
...,...,...,...,...,...,...,...,...,...,...
75,Canada,Qatar,Canada,Cyle Larin,16.0,False,False,False,1,0
76,Canada,Qatar,Canada,Jonathan David,29.0,False,False,False,1,0
77,Canada,Qatar,Canada,Jonathan David,45.0,False,False,False,1,0
78,Canada,Qatar,Canada,Nathan Saliba,64.0,False,False,False,1,0


In [50]:
rank = pd.read_csv(r"fifa_ranking_2026-06-08.csv")
rank.drop(columns=['team_code','association','previous_rank','points','previous_points','rated_matches'] ,inplace=True)
rank.head(3)

,team,rank
0,Argentina,1
1,Spain,2
2,France,3


In [ ]:
def getting_fifa_rank(final_df,rank):
    unique_matches = final_df[['home_team','away_team']].drop_duplicates()

    rank_diff = {}

    for index,row in unique_matches.iterrows():
        home = row['home_team']
        away = row['away_team']

        try:

            home_points = rank.loc[rank['team']==home,'rank'].values[0]
            away_points = rank.loc[rank['team']==away,'rank'].values[0]
            diff =int( home_points-away_points )

            rank_diff[(home , away)] = diff
        except IndexError:
            rank_diff[(home , away)]=0

    return rank_diff


{('Mexico', 'South Africa'): -46,
 ('South Korea', 'Czech Republic'): -14,
 ('Canada', 'Bosnia and Herzegovina'): -34,
 ('United States', 'Paraguay'): -23,
 ('Australia', 'Turkey'): 5,
 ('Brazil', 'Morocco'): -1,
 ('Haiti', 'Scotland'): 41,
 ('Qatar', 'Switzerland'): 38,
 ('Germany', 'Curaçao'): -72,
 ('Ivory Coast', 'Ecuador'): 10,
 ('Netherlands', 'Japan'): -10,
 ('Sweden', 'Tunisia'): -8,
 ('Belgium', 'Egypt'): -20,
 ('Iran', 'New Zealand'): -65,
 ('Saudi Arabia', 'Uruguay'): 45,
 ('Argentina', 'Algeria'): -27,
 ('Austria', 'Jordan'): -39,
 ('France', 'Senegal'): -12,
 ('Iraq', 'Norway'): 25,
 ('England', 'Croatia'): -7,
 ('Ghana', 'Panama'): 39,
 ('Portugal', 'DR Congo'): -40,
 ('Uzbekistan', 'Colombia'): 37,
 ('Canada', 'Qatar'): -27,
 ('Czech Republic', 'South Africa'): -21,
 ('Mexico', 'South Korea'): -11,
 ('Switzerland', 'Bosnia and Herzegovina'): -45}

In [ ]:
rank['team'] = rank['team'].replace({'Korea Republic':'South Korea',
'Czechia':'Czech Republic',
                                     'Türkiye':'Turkey',
                                     'USA':'United States',
                                     "Côte d'Ivoire":'Ivory Coast'
                                     ,'IR Iran':'Iran','Congo DR':'DR Congo'})

rank_diff=getting_fifa_rank(final_df,rank)
team_rank_dict = dict(zip(rank['team'],rank['rank']))
#joblib.dump(team_rank_dict,'team_rank_dict.pkl')


['team_rank_dict.pkl']

In [53]:
#getting the winner 
result = pd.read_csv(r'results.csv')
result.drop(columns=['tournament','city','country','neutral','date'],inplace=True)
result = result[49405:]
result.reset_index(inplace=True,drop=True)
result.dropna(inplace=True)


In [54]:
rules = [
    result['home_score'] > result['away_score'],
    result['away_score'] > result['home_score'],
    result['home_score'] == result['away_score']
]

scores = ['home' ,'away', 'draw']
result['winner'] =  np.select(rules,scores,'draw')
result.drop(columns=['home_score','away_score'],inplace=True)
result[['home_team','away_team']] = result[['home_team','away_team']].replace({'Korea Republic':'South Korea',
                                     'Czechia':'Czech Republic',
                                     'Türkiye':'Turkey',
                                     'USA':'United States',
                                     "Côte d'Ivoire":'Ivory Coast'
                                     ,'IR Iran':'Iran','Congo DR':'DR Congo'})
result

,home_team,away_team,winner
0,Mexico,South Africa,home
1,South Korea,Czech Republic,home
2,Canada,Bosnia and Herzegovina,draw
3,United States,Paraguay,home
4,Qatar,Switzerland,draw
5,Brazil,Morocco,draw
6,Haiti,Scotland,away
7,Australia,Turkey,home
8,Germany,Curaçao,home
9,Ivory Coast,Ecuador,home


In [55]:
X_list = []
y_list = []
for index,row in result.iterrows():
    home = row['home_team']
    away = row['away_team']

    fifa_rank = rank_diff.get((home,away),0)
    ground = final_df[(final_df['home_team']==home) & (final_df['away_team']==away)]
    if not ground.empty:
        neutral = ground['neutral'].values[0]
    else:
        neutral = False
    X_list.append([fifa_rank,neutral])
    y_list.append(row['winner'])

X=np.array(X_list,dtype=object)
y=np.array(y_list)

In [56]:
np.savez('preprocessed_data.npz',X=X,y=y)